In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window


dbutils.widgets.text("catalog","delta_catalog")
dbutils.widgets.text("schema",  "delta_demo")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

# COMMAND ----------
# MAGIC %md ## Read Bronze

bronze_df = spark.table(f"{CATALOG}.{SCHEMA}.bronze_orders")
print("Bronze rows:", bronze_df.count())


# COMMAND ----------
# MAGIC %md ## Filter + Deduplicate
filtered_df=bronze_df.filter("status='completed'")
print("After Filter:", filtered_df.count())

# COMMAND ----------
# MAGIC %md ## Write Bronze

w_dedup=Window.partitionBy("order_id").orderBy("order_date")
deduped_df=filtered_df.withColumn("rn",row_number().over(w_dedup)).filter("rn=1").drop("rn")
print("After Dedup:", deduped_df.count())

# COMMAND ----------
# MAGIC %md ## Coalesce before write — data is smaller now

silver_df=deduped_df.coalesce(4)
print(f"Partitions after coalesce: {silver_df.select(spark_partition_id()).distinct().count()}")




In [0]:
# COMMAND ----------
# MAGIC %md ## Create Silver table with constraints


spark.sql(f"""
          CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.silver_orders (
              order_id bigint not null,
              customer_id int not null,
              product_id int,
              category string,
              amount int,
              order_date DATE,
              status string,
              year int,
              month int
          )
          using delta
          tblproperties( delta.enableChangeDataFeed = true)""")

spark.sql(f"""
          alter table {CATALOG}.{SCHEMA}.silver_orders 
          add constraint amount_check check(amount>=0)
          """)

In [0]:
silver_df.write.format("delta").mode("overwrite")\
    .option("overwriteSchema","true")\
        .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_orders")

print("Silver orders written:",spark.table(f"{CATALOG}.{SCHEMA}.silver_orders").count())
# dbutils.notebook.exit("02_silver_orders: SUCCESS")
